In [4]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType, StringType
from pyspark.sql.functions import from_json, col,to_timestamp, window

Création de la session Spark

In [5]:
spark = SparkSession.builder.appName("ReadClientTickets")\
    .master("local[*]")\
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")\
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/11 16:17:07 WARN Utils: Your hostname, camille-ThinkPad-X1-Carbon-Gen-10, resolves to a loopback address: 127.0.1.1; using 10.59.114.55 instead (on interface wlp0s20f3)
26/08/11 16:17:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/camille_sc/anaconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/camille_sc/.ivy2.5.2/cache
The jars for the packages stored in: /home/camille_sc/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a4b9ebad-3e81-4ec7-b971-40673e9eec28;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#ka

Connexion aux données

In [6]:
df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "client_tickets")
    .option("startingOffsets", "latest")  
    .load()
)


In [7]:
schema = (StructType()
    .add("ticket_id", StringType())
    .add("client_id", StringType())
    .add("created_at", StringType())
    .add("demande", StringType())
    .add("type_demande", StringType())
    .add("priorite", StringType())
)

df_tickets = (
    df.selectExpr("CAST(value AS STRING) AS json_value")
    .select(from_json(col("json_value"), schema).alias("data"))
    .select("data.*")
)
df_tickets_ts = df_tickets.withColumn(
    "created_at_ts", to_timestamp(col("created_at"))
)
df_watermarked = df_tickets_ts.withWatermark("created_at_ts", "1 hour")

Traitement des données

    Nombre de ticket par type de demande

In [8]:
result_1 = (
    df_tickets
    .groupBy("type_demande")   
    .count()
)

Ajouter le nom d'une équipe de support

In [9]:
services = [
    {"type_demande": "Facturation", "equipe_support": "Comptabilité"},
    {"type_demande": "Support technique", "equipe_support": "SAV"},
    {"type_demande": "Résiliation", "equipe_support": "Service Client"},
    {"type_demande": "Question générale", "equipe_support": "Service Client"},
    {"type_demande": "Réclamation", "equipe_support": "Service Client"},
    {"type_demande": "Demande d'information", "equipe_support": "Service Client"},
]
df_services = spark.createDataFrame([Row(**s) for s in services])

df_enrichi = df_tickets.join(df_services, on="type_demande", how="left")

In [10]:
for q in spark.streams.active:
    print(q.name, q.id)


In [ ]:
query_count = (
    result_1.writeStream
    .queryName("comptage_par_type_24h")
    .format("console")
    .outputMode("update")
    .trigger(processingTime="10 seconds")
    .start()
)

query_enrichi = (
    df_enrichi.writeStream
    .queryName("tickets_enrichis")
    .trigger(processingTime="10 seconds")
    .format("console")
    .outputMode("append")
    .start()
)

spark.streams.awaitAnyTermination()

26/08/11 16:17:42 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-ee40e4db-5953-4c51-a1e1-d36c7f3c33d1. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/11 16:17:42 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/08/11 16:17:42 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-85cae371-4e93-4b19-af2f-b562e8ee0623. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/11 16:17:42 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not support

-------------------------------------------
Batch: 0
-------------------------------------------
+------------+-----+
|type_demande|count|
+------------+-----+
+------------+-----+

-------------------------------------------
Batch: 0
-------------------------------------------
+------------+---------+---------+----------+-------+--------+--------------+
|type_demande|ticket_id|client_id|created_at|demande|priorite|equipe_support|
+------------+---------+---------+----------+-------+--------+--------------+
+------------+---------+---------+----------+-------+--------+--------------+



26/08/11 16:18:10 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 27868 milliseconds
26/08/11 16:18:10 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 27877 milliseconds


-------------------------------------------
Batch: 1
-------------------------------------------
-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+-----+
|        type_demande|count|
+--------------------+-----+
|         Réclamation|    1|
|         Facturation|    1|
|Demande d'informa...|    1|
|   Support technique|    2|
|         Résiliation|    3|
|   Question générale|    1|
+--------------------+-----+

+--------------------+--------------------+---------+--------------------+--------------------+--------+--------------+
|        type_demande|           ticket_id|client_id|          created_at|             demande|priorite|equipe_support|
+--------------------+--------------------+---------+--------------------+--------------------+--------+--------------+
|         Réclamation|cce06cbb-6b8e-496...|CUST-0066|2026-08-11T14:17:...|Une fonctionnalit...|   Haute|Service Client|
|         Facturation|cd0622a5-0d52

26/08/11 16:18:23 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 13245 milliseconds
26/08/11 16:18:23 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 13257 milliseconds


...

-------------------------------------------
Batch: 17
-------------------------------------------
+--------------------+-----+
|        type_demande|count|
+--------------------+-----+
|         Réclamation|   17|
|Demande d'informa...|    5|
+--------------------+-----+

-------------------------------------------
Batch: 18
-------------------------------------------
+--------------------+--------------------+---------+--------------------+--------------------+--------+--------------+
|        type_demande|           ticket_id|client_id|          created_at|             demande|priorite|equipe_support|
+--------------------+--------------------+---------+--------------------+--------------------+--------+--------------+
|         Réclamation|b6eda664-818f-426...|CUST-0097|2026-08-11T14:20:...|Comment exporter ...|Critique|Service Client|
|Demande d'informa...|5475e698-c4a3-429...|CUST-0195|2026-08-11T14:20:...|Je souhaite obten...|   Basse|Service Client|
|   Question générale|a7bada9

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------------+-----+
|     type_demande|count|
+-----------------+-----+
|      Réclamation|   18|
|Question générale|    8|
+-----------------+-----+

-------------------------------------------
Batch: 19
-------------------------------------------
+------------+--------------------+---------+--------------------+--------------------+--------+--------------+
|type_demande|           ticket_id|client_id|          created_at|             demande|priorite|equipe_support|
+------------+--------------------+---------+--------------------+--------------------+--------+--------------+
| Réclamation|c351fc0a-51f4-497...|CUST-0100|2026-08-11T14:21:...|J'aimerais mettre...| Moyenne|Service Client|
| Réclamation|cdf9e318-4d8f-441...|CUST-0006|2026-08-11T14:21:...|Comment exporter ...|Critique|Service Client|
+------------+--------------------+---------+--------------------+--------------------+

ERROR:root:KeyboardInterrupt while sending command.             (35 + 16) / 200]
Traceback (most recent call last):
  File "/home/camille_sc/anaconda3/lib/python3.13/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/camille_sc/anaconda3/lib/python3.13/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/home/camille_sc/anaconda3/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

-------------------------------------------
Batch: 19
-------------------------------------------
+------------+-----+
|type_demande|count|
+------------+-----+
| Réclamation|   20|
+------------+-----+

